# Sea Ice Thickness Mean/Std Sweep Diagnostics

This notebook mirrors `experiments_synthetic_eval_sic_mean_std_sweep.ipynb`, but uses sea ice thickness (`CHANNEL = 1`).

The saved `samples/**/*.npz` tensors from `synthetic_eval.cli --save-tensors` are already in physical units. For each candidate pair this notebook does:

```python
thick_norm = (saved_thickness - current_thick_mean) / current_thick_std
thick_candidate = thick_norm * candidate_thick_std + candidate_thick_mean
thick_candidate = clip(thick_candidate, 0, None)
thick_candidate[:, invalid_padding_mask] = 0
```

Computations are optimized so each archive is loaded once per analysis cell while all candidate `mean/std` pairs are evaluated together.


In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})


## 1. Locate The Synthetic Eval Run

This follows `experiments_synthetic_eval_results_viewer.ipynb`: it reads `SYNTH_EVAL_OUT`, otherwise uses `<DATA_ROOT>/synthetic_eval_swath_30days_x32_r4_saved`, otherwise falls back to the newest `synthetic_eval*` directory under `DATA_ROOT`.


In [ ]:
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/mnt/sciml/a.sadreev/sea_ice_data"))
OUT_DIR = Path(
    os.environ.get(
        "SYNTH_EVAL_OUT",
        DATA_ROOT / "synthetic_eval_swath_30days_x32_r4_saved",
    )
)

if not OUT_DIR.exists():
    candidates = sorted(DATA_ROOT.glob("synthetic_eval*"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No synthetic_eval output directories found under {DATA_ROOT}")
    print(f"Configured OUT_DIR does not exist: {OUT_DIR}")
    OUT_DIR = candidates[0]
    print(f"Using latest synthetic_eval directory instead: {OUT_DIR}")

PLOTS_DIR = OUT_DIR / "plots"
ARRAYS_DIR = OUT_DIR / "arrays"
SAMPLES_DIR = OUT_DIR / "samples"
SWEEP_PLOTS_DIR = PLOTS_DIR / "thickness_mean_std_sweep"
SWEEP_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = OUT_DIR / "metadata.json"
metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}

tensor_paths = sorted(SAMPLES_DIR.glob("**/*.npz"))
if not tensor_paths:
    raise FileNotFoundError(f"No saved tensor archives found under {SAMPLES_DIR}; rerun eval with --save-tensors")

print("OUT_DIR    =", OUT_DIR)
print("PLOTS_DIR  =", PLOTS_DIR)
print("ARRAYS_DIR =", ARRAYS_DIR)
print("SAMPLES_DIR=", SAMPLES_DIR)
print("n archives =", len(tensor_paths))
print("eval_region=", metadata.get("eval_region", "unobserved"))
print("rank_stride=", metadata.get("rank_stride", 4))
for path in tensor_paths[:8]:
    print(" ", path.relative_to(OUT_DIR))
if len(tensor_paths) > 8:
    print(f"  ... {len(tensor_paths) - 8} more")


## 2. Current Stats And Candidate Mean/Std Grid

Current stats are loaded from `metadata["stats_json"]`. Change only `THICK_MEAN_STD_GRID` in this cell to test another set of thickness denormalization parameters.


In [ ]:
CHANNEL = 1  # sea ice thickness


def load_channel_stats(metadata):
    stats_json = metadata.get("stats_json")
    if not stats_json:
        raise KeyError("metadata does not contain stats_json")
    stats_path = Path(stats_json)
    if not stats_path.exists():
        raise FileNotFoundError(stats_path)
    with open(stats_path) as f:
        stats = json.load(f)
    return stats_path, np.asarray(stats["mean"], dtype=np.float64), np.asarray(stats["std"], dtype=np.float64)


stats_path, CURRENT_MEAN, CURRENT_STD = load_channel_stats(metadata)
CURRENT_THICK_MEAN = float(CURRENT_MEAN[CHANNEL])
CURRENT_THICK_STD = float(CURRENT_STD[CHANNEL])

print("Current stats_json:", stats_path)
print("Current channel mean:", CURRENT_MEAN)
print("Current channel std: ", CURRENT_STD)
print("Current thickness mean/std:", CURRENT_THICK_MEAN, CURRENT_THICK_STD)

# ============================================================
# CHANGE CANDIDATE THICKNESS MEAN/STD HERE
# Each entry is: (label, candidate_thickness_mean, candidate_thickness_std)
# ============================================================
THICK_MEAN_STD_GRID = [
    ("current", CURRENT_THICK_MEAN, CURRENT_THICK_STD),

    ("mean -0.100", CURRENT_THICK_MEAN - 0.100, CURRENT_THICK_STD),
    ("mean -0.050", CURRENT_THICK_MEAN - 0.050, CURRENT_THICK_STD),
    ("mean -0.025", CURRENT_THICK_MEAN - 0.025, CURRENT_THICK_STD),
    ("mean +0.025", CURRENT_THICK_MEAN + 0.025, CURRENT_THICK_STD),
    ("mean +0.050", CURRENT_THICK_MEAN + 0.050, CURRENT_THICK_STD),
    ("mean +0.100", CURRENT_THICK_MEAN + 0.100, CURRENT_THICK_STD),

    ("std x0.75", CURRENT_THICK_MEAN, CURRENT_THICK_STD * 0.75),
    ("std x0.90", CURRENT_THICK_MEAN, CURRENT_THICK_STD * 0.90),
    ("std x1.10", CURRENT_THICK_MEAN, CURRENT_THICK_STD * 1.10),
    ("std x1.25", CURRENT_THICK_MEAN, CURRENT_THICK_STD * 1.25),

    ("mean -0.025, std x0.90", CURRENT_THICK_MEAN - 0.025, CURRENT_THICK_STD * 0.90),
    ("mean +0.025, std x1.10", CURRENT_THICK_MEAN + 0.025, CURRENT_THICK_STD * 1.10),
]

CANDIDATE_LABELS = [str(x[0]) for x in THICK_MEAN_STD_GRID]
CANDIDATE_MEANS = np.asarray([float(x[1]) for x in THICK_MEAN_STD_GRID], dtype=np.float64)
CANDIDATE_STDS = np.asarray([float(x[2]) for x in THICK_MEAN_STD_GRID], dtype=np.float64)

if np.any(CANDIDATE_STDS <= 0):
    raise ValueError("All candidate std values must be positive")

candidate_table = [
    {"label": label, "thickness_mean": mean, "thickness_std": std}
    for label, mean, std in zip(CANDIDATE_LABELS, CANDIDATE_MEANS, CANDIDATE_STDS)
]
if pd is not None:
    display(pd.DataFrame(candidate_table))
else:
    display(candidate_table)


## 3. Helpers

Invalid padding mask is forced to zero after every candidate transform. Evaluation masks also exclude invalid padding cells. Thickness is clipped to `>= 0` only.


In [ ]:
RANK_STRIDE = int(metadata.get("rank_stride", 4) or 4)
EVAL_REGION = metadata.get("eval_region", "unobserved")


def scalar_int(z, key, default=0):
    if key not in z.files:
        return default
    return int(np.asarray(z[key]).item())


def safe_name(text):
    return (
        str(text)
        .replace("+", "plus")
        .replace("-", "minus")
        .replace(".", "p")
        .replace(", ", "__")
        .replace(" ", "_")
        .replace("=", "")
    )


def load_valid_mask(shape_hw):
    candidates = []
    if metadata.get("mask_path"):
        candidates.append(Path(metadata["mask_path"]))
    candidates.extend([
        DATA_ROOT / "mask_padding.npy",
        Path("/Users/amir/sciml/sea_ice_data/mask_padding.npy"),
        Path("/mnt/sciml/a.sadreev/sea_ice_data/mask_padding.npy"),
    ])
    for path in candidates:
        if path.exists():
            valid = np.load(path).astype(bool)
            if valid.ndim == 3:
                valid = valid[0]
            if tuple(valid.shape) == tuple(shape_hw):
                return valid
    return np.ones(shape_hw, dtype=bool)


def make_eval_mask(observed_mask, valid_mask, eval_region=EVAL_REGION):
    observed = np.asarray(observed_mask, dtype=bool)
    valid = np.asarray(valid_mask, dtype=bool)
    if eval_region == "all":
        return valid
    if eval_region == "observed":
        return valid & observed
    if eval_region == "unobserved":
        return valid & ~observed
    raise ValueError(f"Unknown eval_region: {eval_region}")


def apply_all_candidate_stats(ensemble_thick_physical, valid_mask):
    # Return shape (K, M, H, W), where K is number of candidate mean/std pairs.
    ens_norm = (ensemble_thick_physical.astype(np.float64) - CURRENT_THICK_MEAN) / CURRENT_THICK_STD
    corrected = ens_norm[None, ...] * CANDIDATE_STDS[:, None, None, None] + CANDIDATE_MEANS[:, None, None, None]
    corrected = np.clip(corrected, 0.0, None)

    # Invalid/padding cells must always be zero.
    corrected[:, :, ~valid_mask] = 0.0
    return corrected


def rank_scores_from_counts(counts):
    counts = np.asarray(counts, dtype=np.float64)
    total = counts.sum()
    if total <= 0:
        return {
            "total_ranks": 0,
            "l1_flat": np.nan,
            "edge_ratio_vs_flat": np.nan,
            "center_ratio_vs_flat": np.nan,
            "right_minus_left_edges_in_flat_units": np.nan,
            "chi2_like_not_independent": np.nan,
        }
    probs = counts / total
    expected = 1.0 / len(probs)
    center = probs[len(probs) // 3 : 2 * len(probs) // 3]
    return {
        "total_ranks": int(total),
        "l1_flat": float(np.abs(probs - expected).sum()),
        "edge_ratio_vs_flat": float((probs[0] + probs[-1]) / (2 * expected)),
        "center_ratio_vs_flat": float(center.mean() / expected),
        "right_minus_left_edges_in_flat_units": float((probs[-1] - probs[0]) / expected),
        "chi2_like_not_independent": float(((counts - total * expected) ** 2 / max(total * expected, 1.0)).sum()),
    }


with np.load(tensor_paths[0], allow_pickle=False) as z0:
    truth0 = z0["truth"][CHANNEL]
    valid_mask0 = load_valid_mask(truth0.shape)
    print("sample truth shape:", truth0.shape)
    print("valid fraction:", float(valid_mask0.mean()))
    print("eval region:", EVAL_REGION)
    print("rank stride:", RANK_STRIDE)


## 4. Pixel Value Distributions

This cell builds the pixel-value distribution for every candidate `mean/std`. It loads each archive once and evaluates all candidates together. Invalid padding cells are zeroed before sampling; distributions are drawn over `EVAL_REGION` only.


In [ ]:
MAX_ARCHIVES_FOR_DISTRIBUTION = min(len(tensor_paths), 40)
MAX_PIXELS_PER_ARCHIVE_PER_CANDIDATE = 25000
DISTRIBUTION_SEED = 123

rng = np.random.default_rng(DISTRIBUTION_SEED)
paths_for_dist = tensor_paths[:MAX_ARCHIVES_FOR_DISTRIBUTION]

candidate_values = [[] for _ in CANDIDATE_LABELS]
truth_values = []
raw_saved_values = []
raw_saved_negative = []

t0 = time.perf_counter()
for archive_i, path in enumerate(paths_for_dist):
    with np.load(path, allow_pickle=False) as z:
        ensemble_thick = z["ensemble"][:, CHANNEL]
        truth_thick = z["truth"][CHANNEL].astype(np.float64)
        observed_mask = z["mask"]

    valid_mask = load_valid_mask(truth_thick.shape)
    eval_mask = make_eval_mask(observed_mask, valid_mask, EVAL_REGION)

    corrected = apply_all_candidate_stats(ensemble_thick, valid_mask)
    raw_saved = ensemble_thick.astype(np.float64).copy()
    raw_saved[:, ~valid_mask] = 0.0

    raw_flat = raw_saved[:, eval_mask].reshape(-1)
    raw_saved_negative.append(float(np.mean(raw_flat < 0.0)) if raw_flat.size else np.nan)
    if raw_flat.size > MAX_PIXELS_PER_ARCHIVE_PER_CANDIDATE:
        raw_flat = rng.choice(raw_flat, size=MAX_PIXELS_PER_ARCHIVE_PER_CANDIDATE, replace=False)
    raw_saved_values.append(raw_flat)

    truth_flat = truth_thick[eval_mask].reshape(-1)
    if truth_flat.size > MAX_PIXELS_PER_ARCHIVE_PER_CANDIDATE:
        truth_flat = rng.choice(truth_flat, size=MAX_PIXELS_PER_ARCHIVE_PER_CANDIDATE, replace=False)
    truth_values.append(truth_flat)

    flat_candidates = corrected[:, :, eval_mask].reshape(len(CANDIDATE_LABELS), -1)
    for k in range(len(CANDIDATE_LABELS)):
        vals = flat_candidates[k]
        if vals.size > MAX_PIXELS_PER_ARCHIVE_PER_CANDIDATE:
            vals = rng.choice(vals, size=MAX_PIXELS_PER_ARCHIVE_PER_CANDIDATE, replace=False)
        candidate_values[k].append(vals)

truth_values = np.concatenate(truth_values) if truth_values else np.array([])
raw_saved_values = np.concatenate(raw_saved_values) if raw_saved_values else np.array([])
candidate_values = [np.concatenate(vals) if vals else np.array([]) for vals in candidate_values]

print(f"loaded {len(paths_for_dist)} archives in {time.perf_counter() - t0:.1f} sec")
print("mean raw saved negative fraction:", float(np.nanmean(raw_saved_negative)))

all_for_bins = [truth_values, raw_saved_values] + candidate_values
all_for_bins = np.concatenate([v[np.isfinite(v)] for v in all_for_bins if v.size])
upper = float(np.quantile(all_for_bins, 0.995)) if all_for_bins.size else 1.0
upper = max(upper, 1e-6)
bins = np.linspace(0.0, upper, 81)

ncols = 2
nrows = int(math.ceil(len(CANDIDATE_LABELS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(7.0 * ncols, 2.8 * nrows), squeeze=False, sharex=True)

for ax, label, mean, std, vals in zip(axes.ravel(), CANDIDATE_LABELS, CANDIDATE_MEANS, CANDIDATE_STDS, candidate_values):
    ax.hist(truth_values, bins=bins, density=True, alpha=0.45, color="black", label="truth")
    ax.hist(vals, bins=bins, density=True, alpha=0.60, color="tab:blue", label="candidate ensemble")
    ax.set_title(f"{label}: mean={mean:.4g}, std={std:.4g}")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)

for ax in axes.ravel()[len(CANDIDATE_LABELS):]:
    ax.axis("off")
axes[-1, 0].set_xlabel("sea ice thickness")
if ncols > 1:
    axes[-1, 1].set_xlabel("sea ice thickness")
plt.tight_layout()

out_path = SWEEP_PLOTS_DIR / "thickness_pixel_distributions_by_mean_std.png"
fig.savefig(out_path, dpi=160)
print("saved:", out_path)
plt.show()


## 5. Rank Histograms

This cell computes rank histograms for all candidate `mean/std` pairs. It is optimized to load each archive once and compute all candidate rank arrays in one vectorized block per archive.


In [ ]:
def candidate_rank_counts_for_archive(corrected_sub, truth_sub, rank_mask, seed):
    # corrected_sub shape: (K, M, Hs, Ws); truth_sub shape: (Hs, Ws).
    k_count, ensemble_size = corrected_sub.shape[:2]
    less = np.sum(corrected_sub < truth_sub[None, None, :, :], axis=1)
    ties = np.sum(corrected_sub == truth_sub[None, None, :, :], axis=1)

    rng = np.random.default_rng(seed)
    ranks = less + rng.integers(0, ties + 1)
    ranks = ranks[:, rank_mask]

    out = np.zeros((k_count, ensemble_size + 1), dtype=np.int64)
    for k in range(k_count):
        out[k] = np.bincount(ranks[k].reshape(-1).astype(np.int64), minlength=ensemble_size + 1)
    return out


rank_counts = None
metric_sums = {label: {"rmse_sum": 0.0, "bias_sum": 0.0, "mae_sum": 0.0, "n": 0} for label in CANDIDATE_LABELS}

t0 = time.perf_counter()
for archive_i, path in enumerate(tensor_paths):
    with np.load(path, allow_pickle=False) as z:
        ensemble_thick = z["ensemble"][:, CHANNEL]
        truth_thick = z["truth"][CHANNEL].astype(np.float64)
        observed_mask = z["mask"]
        combo_seed = scalar_int(z, "combo_seed", archive_i)

    valid_mask = load_valid_mask(truth_thick.shape)
    eval_mask = make_eval_mask(observed_mask, valid_mask, EVAL_REGION)

    corrected = apply_all_candidate_stats(ensemble_thick, valid_mask)
    corrected_sub = corrected[:, :, ::RANK_STRIDE, ::RANK_STRIDE]
    truth_sub = truth_thick[::RANK_STRIDE, ::RANK_STRIDE]
    rank_mask = eval_mask[::RANK_STRIDE, ::RANK_STRIDE]

    counts = candidate_rank_counts_for_archive(corrected_sub, truth_sub, rank_mask, seed=combo_seed + CHANNEL)
    rank_counts = counts.copy() if rank_counts is None else rank_counts + counts

    means = corrected.mean(axis=1)  # K, H, W
    diff = means - truth_thick[None, :, :]
    for k, label in enumerate(CANDIDATE_LABELS):
        if np.any(eval_mask):
            metric_sums[label]["rmse_sum"] += float(np.sqrt(np.mean(diff[k][eval_mask] ** 2)))
            metric_sums[label]["bias_sum"] += float(np.mean(diff[k][eval_mask]))
            metric_sums[label]["mae_sum"] += float(np.mean(np.abs(diff[k][eval_mask])))
            metric_sums[label]["n"] += 1

print(f"rank histograms computed for {len(tensor_paths)} archives in {time.perf_counter() - t0:.1f} sec")

rank_rows = []
for k, label in enumerate(CANDIDATE_LABELS):
    scores = rank_scores_from_counts(rank_counts[k])
    n_metric = max(1, metric_sums[label]["n"])
    rank_rows.append({
        "label": label,
        "candidate_mean": float(CANDIDATE_MEANS[k]),
        "candidate_std": float(CANDIDATE_STDS[k]),
        **scores,
        "rmse_mean": metric_sums[label]["rmse_sum"] / n_metric,
        "bias_mean": metric_sums[label]["bias_sum"] / n_metric,
        "mae_mean": metric_sums[label]["mae_sum"] / n_metric,
    })

if pd is not None:
    rank_table = pd.DataFrame(rank_rows).sort_values(["l1_flat", "rmse_mean"])
    display(rank_table)
else:
    rank_table = sorted(rank_rows, key=lambda r: (r["l1_flat"], r["rmse_mean"]))
    display(rank_table)

ncols = 3
nrows = int(math.ceil(len(CANDIDATE_LABELS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.6 * nrows), squeeze=False)

for ax, k, label in zip(axes.ravel(), range(len(CANDIDATE_LABELS)), CANDIDATE_LABELS):
    counts = rank_counts[k].astype(np.float64)
    probs = counts / max(1.0, counts.sum())
    expected = 1.0 / len(probs)
    scores = rank_scores_from_counts(counts)

    ax.bar(np.arange(len(probs)), probs, color="black", width=0.85)
    ax.axhline(expected, color="tab:red", linestyle="--", linewidth=1.2)
    ax.set_title(
        f"{label}\n"
        f"L1={scores['l1_flat']:.3f}, bias={scores['right_minus_left_edges_in_flat_units']:.3f}"
    )
    ax.set_xlabel("rank of truth")
    ax.set_ylabel("probability")

for ax in axes.ravel()[len(CANDIDATE_LABELS):]:
    ax.axis("off")

plt.tight_layout()
out_path = SWEEP_PLOTS_DIR / "thickness_rank_histograms_by_mean_std.png"
fig.savefig(out_path, dpi=160)
print("saved:", out_path)
plt.show()

counts_path = SWEEP_PLOTS_DIR / "thickness_rank_histogram_counts_by_mean_std.npz"
np.savez_compressed(counts_path, **{safe_name(label): rank_counts[k] for k, label in enumerate(CANDIDATE_LABELS)})
print("saved:", counts_path)


## 6. Best Candidate Detail

This cell redraws the best candidate rank histogram as a standalone figure.


In [ ]:
if pd is not None:
    best = rank_table.iloc[0].to_dict()
else:
    best = rank_table[0]

best_label = best["label"]
best_k = CANDIDATE_LABELS.index(best_label)
counts = rank_counts[best_k].astype(np.float64)
probs = counts / max(1.0, counts.sum())
expected = 1.0 / len(probs)

print("best candidate:")
display(best)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(np.arange(len(probs)), probs, color="black", width=0.85)
ax.axhline(expected, color="tab:red", linestyle="--", linewidth=1.2, label="flat")
ax.set_title(
    f"Best thickness mean/std candidate: {best_label}\n"
    f"mean={best['candidate_mean']:.6g}, std={best['candidate_std']:.6g}, "
    f"L1={best['l1_flat']:.3f}, bias={best['right_minus_left_edges_in_flat_units']:.3f}"
)
ax.set_xlabel("rank of truth")
ax.set_ylabel("probability")
ax.legend()
fig.tight_layout()

out_path = SWEEP_PLOTS_DIR / f"thickness_rank_histogram_best__{safe_name(best_label)}.png"
fig.savefig(out_path, dpi=180)
print("saved:", out_path)
plt.show()
